<a href="https://colab.research.google.com/github/ThaiTriNhan/semiconductor-fab-capacity-analyzer/blob/main/FAB_Capacity_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

tools_url = "https://raw.githubusercontent.com/ThaiTriNhan/semiconductor-fab-capacity-analyzer/refs/heads/main/data/raw/tools.csv"

tools = pd.read_csv(tools_url)

tools.head()

tools.shape

(30, 4)

In [6]:
tools

,tool_id,toolset,process,theoretical_wph
0,LITHO_A01,LITHO_A,Lithography,20
1,LITHO_A02,LITHO_A,Lithography,21
2,LITHO_A03,LITHO_A,Lithography,19
3,LITHO_B01,LITHO_B,Lithography,18
4,LITHO_B02,LITHO_B,Lithography,20
5,LITHO_B03,LITHO_B,Lithography,19
6,ETCH_A01,ETCH_A,Etch,35
7,ETCH_A02,ETCH_A,Etch,34
8,ETCH_A03,ETCH_A,Etch,36
9,ETCH_B01,ETCH_B,Etch,32


In [7]:
tools["process"].value_counts()

,count
process,
Lithography,6
Etch,6
Deposition,6
CMP,6
Implant,6


In [8]:
WEEKS = 26

np.random.seed(42)

In [9]:
calendar_records = []

for week in range(1, WEEKS + 1):

    for _, tool in tools.iterrows():

        calendar_records.append({
            "week": week,
            "tool_id": tool["tool_id"],
            "scheduled_hours": 168,
            "pm_hours": np.random.randint(0, 13),
            "downtime_hours": np.random.randint(0, 21)
        })

In [10]:
calendar = pd.DataFrame(calendar_records)

calendar.head()

,week,tool_id,scheduled_hours,pm_hours,downtime_hours
0,1,LITHO_A01,168,6,19
1,1,LITHO_A02,168,12,14
2,1,LITHO_A03,168,10,7
3,1,LITHO_B01,168,12,20
4,1,LITHO_B02,168,6,18


In [11]:
len(calendar)

780

In [12]:
calendar["week"].nunique()

26

In [13]:
calendar["tool_id"].nunique()

30

Bước 3L — Tính Available Hours

Đây là phép tính IE đầu tiên của Project 1.

Công thức:

Available Hours=Scheduled Hours−PM Hours−Downtime Hours

In [14]:
calendar["available_hours"] = (
    calendar["scheduled_hours"]
    - calendar["pm_hours"]
    - calendar["downtime_hours"]
)

In [15]:
calendar.head()

,week,tool_id,scheduled_hours,pm_hours,downtime_hours,available_hours
0,1,LITHO_A01,168,6,19,143
1,1,LITHO_A02,168,12,14,142
2,1,LITHO_A03,168,10,7,151
3,1,LITHO_B01,168,12,20,136
4,1,LITHO_B02,168,6,18,144


In [16]:
calendar = calendar.merge(
    tools[
        [
            "tool_id",
            "toolset",
            "process",
            "theoretical_wph"
        ]
    ],
    on="tool_id",
    how="left"
)

In [17]:
calendar.head()

,week,tool_id,scheduled_hours,pm_hours,downtime_hours,available_hours,toolset,process,theoretical_wph
0,1,LITHO_A01,168,6,19,143,LITHO_A,Lithography,20
1,1,LITHO_A02,168,12,14,142,LITHO_A,Lithography,21
2,1,LITHO_A03,168,10,7,151,LITHO_A,Lithography,19
3,1,LITHO_B01,168,12,20,136,LITHO_B,Lithography,18
4,1,LITHO_B02,168,6,18,144,LITHO_B,Lithography,20


Bước 3N — Tính Effective Capacity

Đây là phần quan trọng nhất của Bước 3.

Công thức:

Effective Capacity=Theoretical WPH×Available Hours

In [18]:
calendar["effective_capacity"] = (
    calendar["theoretical_wph"]
    * calendar["available_hours"]
)

In [19]:
calendar[
    [
        "week",
        "tool_id",
        "toolset",
        "process",
        "theoretical_wph",
        "available_hours",
        "effective_capacity"
    ]
].head(10)

,week,tool_id,toolset,process,theoretical_wph,available_hours,effective_capacity
0,1,LITHO_A01,LITHO_A,Lithography,20,143,2860
1,1,LITHO_A02,LITHO_A,Lithography,21,142,2982
2,1,LITHO_A03,LITHO_A,Lithography,19,151,2869
3,1,LITHO_B01,LITHO_B,Lithography,18,136,2448
4,1,LITHO_B02,LITHO_B,Lithography,20,144,2880
5,1,LITHO_B03,LITHO_B,Lithography,19,152,2888
6,1,ETCH_A01,ETCH_A,Etch,35,138,4830
7,1,ETCH_A02,ETCH_A,Etch,34,158,5372
8,1,ETCH_A03,ETCH_A,Etch,36,159,5724
9,1,ETCH_B01,ETCH_B,Etch,32,143,4576


In [20]:
calendar["effective_capacity"].describe()

,effective_capacity
count,780.000000
mean,3861.594872
std,799.263411
min,2448.000000
25%,3192.000000
50%,3720.000000
75%,4426.250000
max,5976.000000


In [21]:
calendar["effective_capacity"].min()

2448

In [22]:
calendar.to_csv(
    "tool_calendar.csv",
    index=False
)

In [23]:
!ls -lh tool_calendar.csv

-rw-r--r-- 1 root root 38K Aug 25 05:31 tool_calendar.csv


In [24]:
calendar.shape

(780, 10)

In [25]:
calendar.columns

Index(['week', 'tool_id', 'scheduled_hours', 'pm_hours', 'downtime_hours',
       'available_hours', 'toolset', 'process', 'theoretical_wph',
       'effective_capacity'],
      dtype='object')

In [26]:
from google.colab import files

files.download("tool_calendar.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
import shutil

shutil.copy(
    "tool_calendar.csv",
    "/content/drive/MyDrive/tool_calendar.csv"
)

'/content/drive/MyDrive/tool_calendar.csv'

In [29]:
import os
import shutil

folder = "/content/drive/MyDrive/FAB_Capacity_Analyzer"

os.makedirs(folder, exist_ok=True)

shutil.copy(
    "tool_calendar.csv",
    f"{folder}/tool_calendar.csv"
)

'/content/drive/MyDrive/FAB_Capacity_Analyzer/tool_calendar.csv'